# Sandbox for Spectral Feature Development

In [6]:
#To set the correct root for importing libraries
import sys
import os
import pandas as pd
import numpy as np

root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(root)

print(root)

/Users/ethanparks/Desktop/repos/spectral-agent-topologies


In [7]:
from src.spectral.spanalysis import (
    Spectral, 
    chain_adjacency, 
    star_adjacency, 
    mesh_adjacency
)

#Official experimental parameters
GAMMA = 0.9
N_STEPS = 12
N_LEAVES = 4

In [8]:
#Instantiates topologies and their adjacency matrices
topologies = {
    "chain": chain_adjacency(N_STEPS),
    "star":  star_adjacency(N_LEAVES),
    "mesh":  mesh_adjacency(N_LEAVES)
}

spectral_data = []

#Performs spectral analysis for each topology
for name, adj in topologies.items():
    analyzer = Spectral(adj, gamma=GAMMA)
    metrics = analyzer.full_single_analysis()
    
    spectral_data.append({
        "Topology": name,
        "Radius (Stability)": metrics.spectral_radius,
        "Gap (Convergence)": metrics.spectral_gap,
        "Condition # (Robustness)": metrics.condition_number
    })


df = pd.DataFrame(spectral_data)
print("--- Theoretical Spectral Metrics ---")
print(df.to_string(index=False))


def get_rank(column, ascending=True):
    sorted_df = df.sort_values(by=column, ascending=ascending)
    return tuple(sorted_df['Topology'].tolist())

#Order of topology based on spectral score for Spearman rank correlation
print("\n--- PREDICTED_ORDER in analysis.py ---")
print(f"Stability (Lower is better):   {get_rank('Radius (Stability)', True)}")
print(f"Convergence (Higher is better): {get_rank('Gap (Convergence)', False)}")
print(f"Robustness (Lower is better):  {get_rank('Condition # (Robustness)', True)}")

--- Theoretical Spectral Metrics ---
Topology  Radius (Stability)  Gap (Convergence)  Condition # (Robustness)
   chain                 1.0           0.000000                  9.945149
    star                10.0           9.000000                 28.609784
    mesh                10.0           9.230769                 13.000000

--- PREDICTED_ORDER in analysis.py ---
Stability (Lower is better):   ('chain', 'mesh', 'star')
Convergence (Higher is better): ('mesh', 'star', 'chain')
Robustness (Lower is better):  ('chain', 'mesh', 'star')


## Matrix Visualization

In [ ]:

print(f"--- Spectral Metrics (γ={GAMMA}) ---\n")

for name, A in topologies.items():
    print("="*80)
    print(f"Structure: {name}")
    print("="*80)

    #To compute the transition matrix
    print("--- Transition Matrix ---")
    print(np.round(A, 3))
    print(f"Transition Matrix Shape: {A.shape[0]}x{A.shape[1]}")

    #To compute the SR and spectral features
    spec = Spectral(A, gamma=GAMMA)
    res = spec.full_single_analysis()

    print("--- SR Matrix ---")
    print(np.round(res.sr_matrix, 3))

    #Sorting eigenvalues by magnitude to show Spectral Radius clearly
    sorted_eigs = np.sort(np.abs(res.eigenvalues))[::-1]

    print("--- Spectral Metrics (The Three Pillars) ---")
    print(f"Sorted Eigs: {sorted_eigs}")
    print(f"Spectral Radius (Stability): {res.spectral_radius}")
    print(f"Spectral Gap (Convergence): {res.spectral_gap}")
    print(f"Condition Number (Robustness): {res.condition_number}")



--- Spectral Metrics (γ=0.9) ---

Structure: chain
--- Transition Matrix ---
[[0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]]
Transition Matrix Shape: 12x12
--- SR Matrix ---
[[1.    0.9   0.81  0.729 0.656 0.59  0.531 0.478 0.43  0.387 0.349 0.314]
 [0.    1.    0.9   0.81  0.729 0.656 0.59  0.531 0.478 0.43  0.387 0.349]
 [0.    0.    1.    0.9   0.81  0.729 0.656 0.59  0.531 0.478 0.43  0.387]
 [0.    0.    0.    1.    0.9   0.81  0.729 0.656 0.59  0.531 0.478 0.43 ]
 [0.    0.    0.    0.    1.    0.9   0.81  0.729 0.656 0.59  0.531 0.478]
 [0.    0.    0.    0.    0.  